**Explanation Needs Identifier** </br></br>
The Explanation Needs Identifier module develops the critical skill of recognizing when and why AI explanations are truly needed versus when they add little value. Through interactive scenarios, users practice evaluating situations based on key factors: decision stakes, user agency (can they act on the explanation?), trust levels, learning opportunities, contestability needs, and regulatory requirements. Each scenario presents an AI decision, and users must determine whether an explanation is essential, valuable, optional, or unnecessary—and identify the primary reason why. This builds the practical judgment needed to allocate explanation resources effectively rather than defaulting to explaining everything or nothing.

In [ ]:
#@title pip install
from IPython.display import clear_output

!pip install ipywidgets
clear_output()  # Hides the output after the cell runs

In [ ]:
#@title Imports
import random
import time
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import Button, VBox, HBox, HTML, Layout, RadioButtons, Checkbox

In [ ]:
#@title Explanation Need Framework
need_categories = {
    "Essential": {
        "description": "Explanation is absolutely required",
        "typical_reasons": "High stakes, legal requirement, contestable decision, trust-critical"
    },
    "Valuable": {
        "description": "Explanation provides clear benefit",
        "typical_reasons": "Learning opportunity, actionable insights, trust-building, user agency"
    },
    "Optional": {
        "description": "Explanation available but not necessary",
        "typical_reasons": "Low stakes, users trust system, curiosity only, minimal actionability"
    },
    "Unnecessary": {
        "description": "Explanation adds little to no value",
        "typical_reasons": "Trivial decision, no user action possible, well-trusted system, distraction"
    }
}

explanation_drivers = {
    "High Stakes": "Decision has significant impact on user's life, safety, or resources",
    "Contestability": "User needs to challenge or appeal the decision",
    "Actionability": "User can take action based on understanding the explanation",
    "Learning": "Explanation helps user understand system and make better decisions",
    "Trust Building": "Explanation increases confidence in AI system",
    "Legal/Regulatory": "Law or regulation requires explanation",
    "Debugging/Improvement": "Developers need to understand system behavior",
    "Safety": "Understanding is critical for safe use"
}

scenarios = [
    {
        "title": "Loan Rejection",
        "context": "Bank's AI denies a small business loan application.",
        "ai_decision": "Application denied - insufficient collateral and credit history",
        "stakes": "High - affects business viability",
        "user_can_act": "Yes - can improve credit, provide collateral, or appeal",
        "trust_level": "Low - financial decisions are inherently distrusted",
        "frequency": "Rare - major life decision",
        "correct_need": "Essential",
        "primary_drivers": ["High Stakes", "Contestability", "Actionability", "Legal/Regulatory"],
        "reasoning": "High-stakes financial decision with legal requirements (adverse action notice), user can contest or improve application, and clear actionability. Explanation is legally and ethically required."
    },
    {
        "title": "Medical Diagnosis Support",
        "context": "AI flags potential cancer in radiology scan for doctor review.",
        "ai_decision": "Suspicious mass detected - recommend biopsy",
        "stakes": "Extremely high - patient health and life",
        "user_can_act": "Yes - doctor makes treatment decisions",
        "trust_level": "Medium - doctors need to verify AI reasoning",
        "frequency": "Regular - part of clinical workflow",
        "correct_need": "Essential",
        "primary_drivers": ["High Stakes", "Safety", "Actionability", "Debugging/Improvement"],
        "reasoning": "Life-or-death stakes require doctors to understand AI reasoning to make sound medical judgments. Safety-critical context demands transparency for clinical decision-making."
    },
    {
        "title": "Netflix Movie Recommendation",
        "context": "Streaming service suggests a romantic comedy to user.",
        "ai_decision": "Recommended: 'Love Actually' because you watched similar films",
        "stakes": "Very low - entertainment only",
        "user_can_act": "Minimal - can only accept or reject",
        "trust_level": "High - users generally trust entertainment recommendations",
        "frequency": "Constant - hundreds of recommendations",
        "correct_need": "Optional",
        "primary_drivers": ["Learning"],
        "reasoning": "Very low stakes and high frequency make explanations optional. Some users enjoy understanding why they get recommendations (learning), but most just want to watch. Brief explanation ('because you watched X') is sufficient when provided."
    },
    {
        "title": "Autocomplete Suggestions",
        "context": "Search engine offers autocomplete suggestions as user types.",
        "ai_decision": "Suggests 'machine learning tutorial' when user types 'machine l'",
        "stakes": "None - trivial convenience feature",
        "user_can_act": "Irrelevant - user just picks or ignores",
        "trust_level": "High - users expect this to work",
        "frequency": "Extreme - happens constantly",
        "correct_need": "Unnecessary",
        "primary_drivers": [],
        "reasoning": "Explaining autocomplete would be absurd - it's trivial, instantaneous, non-binding, and happens thousands of times. Users don't need or want to know why 'machine l' suggests 'machine learning'. Explanation would be pure distraction."
    },
    {
        "title": "Resume Screening Rejection",
        "context": "HR AI rejects job application during initial screening.",
        "ai_decision": "Application does not meet minimum qualifications",
        "stakes": "High - affects livelihood and career",
        "user_can_act": "Yes - can improve resume, gain skills, or challenge decision",
        "trust_level": "Very low - hiring is high-stakes and often opaque",
        "frequency": "Rare for individual, common for company",
        "correct_need": "Essential",
        "primary_drivers": ["High Stakes", "Contestability", "Actionability", "Trust Building"],
        "reasoning": "High-stakes decision affecting livelihood. Candidates need to understand rejections to improve (actionability) and should be able to contest errors (contestability). Explanation builds trust in hiring process."
    },
    {
        "title": "Smart Thermostat Adjustment",
        "context": "Smart thermostat automatically lowers temperature by 1 degree.",
        "ai_decision": "Reduced temperature from 72°F to 71°F based on occupancy",
        "stakes": "Very low - minor comfort and savings",
        "user_can_act": "Yes - can override easily",
        "trust_level": "High - users generally trust smart home devices",
        "frequency": "Very frequent - happens daily",
        "correct_need": "Optional",
        "primary_drivers": ["Learning", "Trust Building"],
        "reasoning": "Low stakes and easy override make explanation optional. Brief explanation ('No one home, saving energy') can build trust and help users learn system behavior, but detailed explanation isn't needed for a 1-degree change."
    },
    {
        "title": "Fraud Detection Block",
        "context": "Credit card system blocks transaction as potential fraud.",
        "ai_decision": "Transaction blocked - unusual pattern detected",
        "stakes": "Medium-High - inconvenience and potential emergencies",
        "user_can_act": "Yes - can confirm legitimate and unblock",
        "trust_level": "Medium - users understand need for security",
        "frequency": "Rare for individual user",
        "correct_need": "Valuable",
        "primary_drivers": ["Trust Building", "Actionability", "Learning"],
        "reasoning": "Users need enough explanation to understand the block and confirm legitimacy. Brief explanation ('Purchase from new country') helps trust and action. Not 'essential' because users accept some security friction, but definitely valuable to reduce frustration."
    },
    {
        "title": "Social Media Feed Ranking",
        "context": "Social platform decides which posts appear at top of feed.",
        "ai_decision": "Post from friend Sarah shown first, post from brand shown later",
        "stakes": "Low - entertainment and information",
        "user_can_act": "Limited - can adjust preferences somewhat",
        "trust_level": "Variable - depends on platform and user",
        "frequency": "Constant - hundreds of ranking decisions",
        "correct_need": "Optional",
        "primary_drivers": ["Trust Building", "Learning"],
        "reasoning": "Low stakes and high frequency make detailed explanations impractical. Brief indicators ('You interact with Sarah frequently') can build trust and help users understand the platform, but comprehensive explanations would overwhelm the experience."
    },
    {
        "title": "Email Spam Filter",
        "context": "Email client automatically filters message to spam folder.",
        "ai_decision": "Email from unknown sender flagged as spam",
        "stakes": "Low-Medium - risk of missing important email",
        "user_can_act": "Yes - can mark as not spam and correct filter",
        "trust_level": "High - users trust spam filters",
        "frequency": "Very frequent - happens constantly",
        "correct_need": "Optional",
        "primary_drivers": ["Learning", "Trust Building"],
        "reasoning": "Most spam filtering should be invisible. Brief explanation ('Unknown sender with suspicious links') when users check spam can help them trust the system and learn to recognize spam. Full explanation unnecessary for routine filtering."
    },
    {
        "title": "Autonomous Vehicle Braking",
        "context": "Self-driving car applies brakes suddenly to avoid pedestrian.",
        "ai_decision": "Emergency braking initiated - pedestrian detected",
        "stakes": "Extremely high - safety critical",
        "user_can_act": "No - emergency response is automatic",
        "trust_level": "Critical - safety systems must be trusted",
        "frequency": "Rare - emergency situations",
        "correct_need": "Valuable",
        "primary_drivers": ["Safety", "Trust Building", "Debugging/Improvement"],
        "reasoning": "Post-event explanation ('Pedestrian entered roadway') builds trust and helps passengers understand safety systems. Not 'essential' because no user action needed in the moment, but valuable for trust and for engineers to verify system behavior. Real-time explanation during emergency would be distracting."
    },
    {
        "title": "Grammarly Writing Suggestion",
        "context": "AI writing assistant suggests changing 'your' to 'you're'.",
        "ai_decision": "Grammar correction suggested",
        "stakes": "Very low - minor writing improvement",
        "user_can_act": "Yes - can accept or reject easily",
        "trust_level": "High - users expect grammar tools to work",
        "frequency": "Very frequent - constant suggestions",
        "correct_need": "Valuable",
        "primary_drivers": ["Learning", "Actionability"],
        "reasoning": "Brief explanation ('Contraction of you are requires apostrophe') helps users learn grammar rules and make informed decisions. Low stakes but high learning value. Explanation should be concise and optional (hover/click to see details)."
    },
    {
        "title": "Prison Recidivism Risk Score",
        "context": "AI calculates risk score used in parole hearing decisions.",
        "ai_decision": "High risk score (78/100) for parole candidate",
        "stakes": "Extremely high - affects freedom and future",
        "user_can_act": "Yes - can contest factors, present counter-evidence",
        "trust_level": "Very low - criminal justice AI is controversial",
        "frequency": "Rare - major life decision",
        "correct_need": "Essential",
        "primary_drivers": ["High Stakes", "Contestability", "Legal/Regulatory", "Trust Building"],
        "reasoning": "Highest possible stakes (human freedom), legal requirements for due process, absolute need for contestability. Explanation must be detailed, defensible, and allow for meaningful challenge. This is a quintessential case where explanation is non-negotiable."
    }
]

In [ ]:
#@title Explanation Needs Identifier
class ExplanationNeedsIdentifier:
    def __init__(self, scenarios_list, categories_dict, drivers_dict):
        self.scenarios = scenarios_list.copy()
        self.categories = categories_dict
        self.drivers = drivers_dict
        self.current_scenario = 0
        self.score = 0
        self.selections = []
        
        random.shuffle(self.scenarios)
        
        # Create widgets
        self.scenario_display = HTML(value="<h3>Explanation Needs Identifier</h3>")
        
        self.need_selector = RadioButtons(
            options=list(self.categories.keys()),
            description='Need Level:',
            disabled=False,
            layout=Layout(width='100%')
        )
        
        self.category_info = HTML(value="")
        
        self.drivers_display = HTML(value="<h4>Why is explanation needed? (Select all that apply)</h4>")
        
        self.driver_checkboxes = []
        for driver in self.drivers.keys():
            cb = Checkbox(
                value=False,
                description=driver,
                disabled=False,
                indent=False,
                layout=Layout(width='100%')
            )
            self.driver_checkboxes.append(cb)
        
        self.submit_button = Button(
            description='Submit Analysis',
            button_style='primary',
            layout=Layout(width='200px', height='40px', margin='10px')
        )
        
        self.feedback_display = HTML(value="")
        
        self.next_button = Button(
            description='Next Scenario →',
            button_style='',
            layout=Layout(width='150px', height='30px', margin='10px')
        )
        
        self.progress_display = HTML(value="")
        
        # Events
        self.need_selector.observe(self.show_category_info, names='value')
        self.submit_button.on_click(lambda b: self.submit_analysis())
        self.next_button.on_click(lambda b: self.next_scenario())
        
        # Layout
        drivers_box = VBox(self.driver_checkboxes, layout=Layout(margin='10px 0'))
        submit_box = HBox([self.submit_button], layout=Layout(justify_content='center'))
        next_box = HBox([self.next_button], layout=Layout(justify_content='center'))
        
        self.widget = VBox([
            self.scenario_display,
            self.need_selector,
            self.category_info,
            self.drivers_display,
            drivers_box,
            submit_box,
            self.feedback_display,
            next_box,
            self.progress_display
        ])
    
    def start_identifier(self):
        self.current_scenario = 0
        self.score = 0
        self.selections = []
        self.show_scenario()
        return self.widget
    
    def show_scenario(self):
        if self.current_scenario >= len(self.scenarios):
            self.show_results()
            return
        
        scenario = self.scenarios[self.current_scenario]
        
        self.scenario_display.value = f"""
        <div style='padding: 20px; background: #f8f9fa; border-radius: 10px; margin: 15px 0;'>
            <h3 style='color: #2c3e50; margin-bottom: 15px;'>Scenario {self.current_scenario + 1}: {scenario['title']}</h3>
            <div style='background: white; padding: 15px; border-radius: 8px; margin-bottom: 15px;'>
                <p style='margin: 0 0 10px 0;'><strong>Context:</strong> {scenario['context']}</p>
                <p style='margin: 0 0 10px 0;'><strong>AI Decision:</strong> {scenario['ai_decision']}</p>
                <p style='margin: 0 0 10px 0;'><strong>Stakes:</strong> {scenario['stakes']}</p>
                <p style='margin: 0 0 10px 0;'><strong>User Can Act:</strong> {scenario['user_can_act']}</p>
                <p style='margin: 0 0 10px 0;'><strong>Trust Level:</strong> {scenario['trust_level']}</p>
                <p style='margin: 0;'><strong>Frequency:</strong> {scenario['frequency']}</p>
            </div>
            <div style='background: #fff3cd; padding: 12px; border-radius: 8px;'>
                <p style='color: #856404; margin: 0; font-weight: bold;'>🔍 Is an explanation needed for this AI decision? How much? Why?</p>
            </div>
        </div>
        """
        
        # Reset
        self.need_selector.value = list(self.categories.keys())[0]
        for cb in self.driver_checkboxes:
            cb.value = False
        self.feedback_display.value = ""
        self.next_button.layout.display = 'none'
        self.submit_button.disabled = False
        self.need_selector.disabled = False
        for cb in self.driver_checkboxes:
            cb.disabled = False
        
        self.show_category_info({'new': self.need_selector.value})
        self.update_progress()
    
    def show_category_info(self, change):
        category = change['new']
        if category in self.categories:
            info = self.categories[category]
            
            self.category_info.value = f"""
            <div style='border: 2px solid #6c757d; padding: 12px; border-radius: 8px; margin: 10px 0; background: #f8f9fa;'>
                <p style='margin: 0 0 5px 0;'><strong>{category}:</strong> {info['description']}</p>
                <p style='margin: 0; font-size: 13px; color: #666;'><em>Typical reasons: {info['typical_reasons']}</em></p>
            </div>
            """
    
    def submit_analysis(self):
        scenario = self.scenarios[self.current_scenario]
        selected_need = self.need_selector.value
        correct_need = scenario['correct_need']
        
        # Get selected drivers
        selected_drivers = [cb.description for cb in self.driver_checkboxes if cb.value]
        correct_drivers = scenario['primary_drivers']
        
        # Score need level
        need_correct = selected_need == correct_need
        need_score = 2 if need_correct else 0
        
        # Score drivers (partial credit)
        if len(correct_drivers) > 0:
            matching_drivers = len(set(selected_drivers) & set(correct_drivers))
            driver_accuracy = matching_drivers / len(correct_drivers)
            driver_score = int(driver_accuracy * 2)
        else:
            # For "Unnecessary" scenarios with no drivers
            driver_score = 2 if len(selected_drivers) == 0 else 0
        
        total_score = need_score + driver_score
        self.score += total_score
        
        self.selections.append({
            'scenario': scenario['title'],
            'selected_need': selected_need,
            'correct_need': correct_need,
            'need_correct': need_correct,
            'score': total_score
        })
        
        # Feedback
        if need_correct and driver_score >= 1:
            color = "#27ae60"
            result = "✓ Excellent Analysis!"
        elif need_correct:
            color = "#f39c12"
            result = "~ Good Need Assessment, Check Drivers"
        else:
            color = "#e74c3c"
            result = "✗ Reconsider the Need Level"
        
        missed_drivers = set(correct_drivers) - set(selected_drivers)
        extra_drivers = set(selected_drivers) - set(correct_drivers)
        
        drivers_feedback = ""
        if len(correct_drivers) > 0:
            drivers_feedback = f"<p><strong>Key drivers:</strong> {', '.join(correct_drivers)}</p>"
            if missed_drivers:
                drivers_feedback += f"<p style='color: #e67e22;'><strong>You missed:</strong> {', '.join(missed_drivers)}</p>"
            if extra_drivers:
                drivers_feedback += f"<p style='color: #95a5a6;'><strong>Less relevant:</strong> {', '.join(extra_drivers)}</p>"
        else:
            drivers_feedback = "<p><strong>No significant drivers</strong> - explanation adds little value</p>"
        
        self.feedback_display.value = f"""
        <div style='border: 2px solid {color}; padding: 20px; border-radius: 10px; margin: 15px 0; background: {color}15;'>
            <h4 style='color: {color}; margin: 0 0 15px 0;'>{result}</h4>
            <p><strong>Correct Need Level:</strong> {correct_need}</p>
            <p><strong>Why:</strong> {scenario['reasoning']}</p>
            {drivers_feedback}
            <p style='margin-top: 15px;'><strong>Your Score:</strong> {total_score}/4 points</p>
        </div>
        """
        
        self.submit_button.disabled = True
        self.need_selector.disabled = True
        for cb in self.driver_checkboxes:
            cb.disabled = True
        self.next_button.layout.display = 'inline-block'
        self.update_progress()
    
    def next_scenario(self):
        self.current_scenario += 1
        self.show_scenario()
    
    def update_progress(self):
        total = len(self.scenarios)
        completed = len(self.selections)
        max_score = completed * 4
        accuracy = (self.score / max_score * 100) if max_score > 0 else 0
        
        self.progress_display.value = f"""
        <div style='background: #e8f4fd; padding: 12px; border-radius: 8px; text-align: center; margin-top: 15px;'>
            <strong>Progress:</strong> {completed}/{total} scenarios |
            <strong>Score:</strong> {self.score}/{max_score} points |
            <strong>Accuracy:</strong> {accuracy:.0f}%
        </div>
        """
    
    def show_results(self):
        total = len(self.scenarios)
        max_score = total * 4
        accuracy = (self.score / max_score * 100)
        perfect_count = sum(1 for s in self.selections if s['score'] == 4)
        
        self.scenario_display.value = f"""
        <div style='border: 3px solid #4CAF50; padding: 30px; border-radius: 15px; background: #f8fff8; text-align: center;'>
            <h2 style='color: #4CAF50; margin: 0 0 20px 0;'>Explanation Needs Identification Complete! 🔍</h2>
            
            <div style='margin: 20px 0;'>
                <h3>Your Results</h3>
                <p style='font-size: 20px;'><strong>Final Score:</strong> {self.score}/{max_score} points ({accuracy:.0f}%)</p>
                <p><strong>Perfect Analyses:</strong> {perfect_count}/{total}</p>
            </div>
            
            <h3>Skill Developed: Explanation Need Recognition</h3>
            <ul style='text-align: left; max-width: 700px; margin: 20px auto;'>
                <li><strong>Context Evaluation:</strong> Learned to assess stakes, actionability, and frequency</li>
                <li><strong>Resource Allocation:</strong> Practiced determining where explanation effort is most valuable</li>
                <li><strong>Driver Identification:</strong> Recognized why explanations matter in different situations</li>
                <li><strong>Avoiding Over-explanation:</strong> Understood when explanations add little value</li>
            </ul>
            
            <h4>Key Insight</h4>
            <p style='max-width: 700px; margin: 0 auto;'>Not every AI decision needs explanation. Effective transparency means
            recognizing when explanation is essential (high stakes, contestable, safety-critical), valuable (builds trust,
            enables learning, supports action), optional (low stakes but potentially interesting), or unnecessary (trivial,
            frequent, no user benefit). This judgment skill lets you allocate explanation resources where they matter most.</p>
        </div>
        """
        
        self.need_selector.layout.display = 'none'
        self.category_info.value = ""
        self.drivers_display.value = ""
        for cb in self.driver_checkboxes:
            cb.layout.display = 'none'
        self.submit_button.layout.display = 'none'
        self.feedback_display.value = ""
        self.next_button.layout.display = 'none'

In [ ]:
#@title Start the Module
# Start the Explanation Needs Identifier
identifier = ExplanationNeedsIdentifier(scenarios, need_categories, explanation_drivers)
widget = identifier.start_identifier()
display(widget)